In [1]:
# CELL 1: Install
!pip install scikit-learn pandas numpy matplotlib seaborn -q
print("✅ Installation complete!")

✅ Installation complete!


In [2]:
# CELL 2: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings
warnings.filterwarnings('ignore')
print("✅ Imports done!")

✅ Imports done!


In [3]:
# CELL 3: Load your dataset
df = pd.read_csv('dataset.csv')  # ← YOUR FILE NAME

print(f"Loaded {len(df)} total reviews")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded 60889 total reviews
Columns: ['Unique_ID', 'Category', 'Review_Header', 'Review_text', 'Rating', 'Own_Rating']


,Unique_ID,Category,Review_Header,Review_text,Rating,Own_Rating
0,136040,smartTv,Nice one,I liked it,5,Positive
1,134236,mobile,Huge battery life with amazing display,I bought the phone on Amazon and been using my...,5,Positive
2,113945,books,Four Stars,"Awesome book at reasonable price, must buy ......",4,Positive


In [4]:
# CELL 4: STRONG DATA CLEANING - Fixes NaN error
print("=== STRONG DATA CLEANING ===\n")

# Check before
print(f"Before cleaning: {len(df)} reviews")

# 1. Remove rows with NaN in Review_text
df = df.dropna(subset=['Review_text'])

# 2. Convert to string and remove empty/whitespace only
df['Review_text'] = df['Review_text'].astype(str)
df = df[df['Review_text'].str.strip() != '']
df = df[df['Review_text'] != 'nan']
df = df[df['Review_text'] != 'NaN']

# 3. Remove rows with NaN in Own_Rating or Rating
df = df.dropna(subset=['Own_Rating', 'Rating', 'Category'])

# 4. Map emotions
emotion_mapping = {'Positive': 0, 'Neutral': 1, 'Negative': 2}
df['label'] = df['Own_Rating'].map(emotion_mapping)

# 5. Remove unmapped rows
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# 6. Final check - Convert everything to string again
df['Review_text'] = df['Review_text'].astype(str).fillna('')

print(f"After cleaning: {len(df)} reviews")
print(f"\n✅ No NaN values left: {not df['Review_text'].isna().any()}")
print(f"\nClass distribution:")
print(df['Own_Rating'].value_counts())

=== STRONG DATA CLEANING ===

Before cleaning: 60889 reviews
After cleaning: 60857 reviews

✅ No NaN values left: True

Class distribution:
Own_Rating
Positive    47407
Negative     9086
Neutral      4364
Name: count, dtype: int64


In [5]:
# CELL 5: Verify no NaN left before TF-IDF
print("=== FINAL VERIFICATION ===\n")

# Double check - convert to list and check each value
X_list = df['Review_text'].tolist()
print(f"Total reviews: {len(X_list)}")

# Check for any problematic values
problematic = []
for i, text in enumerate(X_list):
    if pd.isna(text) or text is None or text == '' or text == 'nan':
        problematic.append(i)

if problematic:
    print(f"⚠️ Found {len(problematic)} problematic reviews - removing them")
    df = df.drop(index=problematic)
    X_list = df['Review_text'].tolist()
else:
    print("✅ All reviews are clean and valid")

print(f"Final dataset size: {len(df)} reviews")

=== FINAL VERIFICATION ===

Total reviews: 60857
✅ All reviews are clean and valid
Final dataset size: 60857 reviews


In [8]:
# CELL 6: Feature engineering (TF-IDF)
print("=== FEATURE ENGINEERING ===\n")

X = df['Review_text'].values
y = df['label'].values

# Convert to string array explicitly
X = np.array([str(text) for text in X])

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    lowercase=True
)

X_features = vectorizer.fit_transform(X)

print(f"✅ Converted {X.shape[0]} reviews to {X_features.shape[1]} features")
print(f"   Feature matrix shape: {X_features.shape}")

=== FEATURE ENGINEERING ===

✅ Converted 60857 reviews to 5000 features
   Feature matrix shape: (60857, 5000)


In [9]:
# CELL 7: Train-test split (FIXED)
print("=== TRAIN-TEST SPLIT ===\n")

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

# Fixed: Use .shape[0] for sparse matrices
print(f"✅ Training set: {X_train.shape[0]} samples")
print(f"✅ Testing set: {X_test.shape[0]} samples")

=== TRAIN-TEST SPLIT ===

✅ Training set: 48685 samples
✅ Testing set: 12172 samples


In [10]:
# CELL 8: Train models (FIXED)
print("="*60)
print("🤖 TRAINING AI MODELS")
print("="*60)

# Only train if we have enough samples
if X_train.shape[0] > 0 and X_test.shape[0] > 0:
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Naive Bayes': MultinomialNB()
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        results[name] = {'model': model, 'accuracy': accuracy}
        print(f"   ✅ Accuracy: {accuracy:.2%}")
    
    # Best model
    best_name = max(results, key=lambda x: results[x]['accuracy'])
    best_model = results[best_name]['model']
    best_acc = results[best_name]['accuracy']
    
    print(f"\n{'='*60}")
    print(f"🏆 BEST MODEL: {best_name}")
    print(f"📊 Accuracy: {best_acc:.2%}")
    
    # Save model
    joblib.dump(best_model, 'emotion_model.pkl')
    joblib.dump(vectorizer, 'vectorizer.pkl')
    print(f"\n✅ Model saved to 'emotion_model.pkl'")
    print(f"✅ Vectorizer saved to 'vectorizer.pkl'")
else:
    print("❌ Not enough data to train models")
    print(f"   Training samples: {X_train.shape[0]}")
    print(f"   Testing samples: {X_test.shape[0]}")

🤖 TRAINING AI MODELS

Training Logistic Regression...
   ✅ Accuracy: 86.44%

Training Random Forest...
   ✅ Accuracy: 85.34%

Training Naive Bayes...
   ✅ Accuracy: 85.98%

🏆 BEST MODEL: Logistic Regression
📊 Accuracy: 86.44%

✅ Model saved to 'emotion_model.pkl'
✅ Vectorizer saved to 'vectorizer.pkl'


In [11]:
# CELL 9: Prediction function
def predict_emotion(review_text):
    if not review_text or str(review_text).strip() == '':
        return "unknown", 0.0, "⚠️ Empty review"
    
    review_features = vectorizer.transform([str(review_text)])
    prediction = best_model.predict(review_features)[0]
    probabilities = best_model.predict_proba(review_features)[0]
    confidence = max(probabilities)
    
    emotions = {0: 'joy/happy', 1: 'neutral', 2: 'anger/sadness'}
    
    if prediction == 2:
        action = "🔴 CREATE HIGH PRIORITY TICKET"
    elif prediction == 1:
        action = "🟡 CREATE MEDIUM PRIORITY TICKET"
    else:
        action = "🟢 NO TICKET NEEDED"
    
    return emotions[prediction], confidence, action

# Test
print("Testing prediction function:\n")
test_reviews = [
    "I love this product!",
    "This is terrible, I want a refund",
    "The product is okay"
]
for review in test_reviews:
    emotion, conf, action = predict_emotion(review)
    print(f"Review: {review}")
    print(f"→ {emotion} ({conf:.2%}) - {action}\n")

Testing prediction function:

Review: I love this product!
→ joy/happy (98.51%) - 🟢 NO TICKET NEEDED

Review: This is terrible, I want a refund
→ anger/sadness (88.53%) - 🔴 CREATE HIGH PRIORITY TICKET

Review: The product is okay
→ joy/happy (51.29%) - 🟢 NO TICKET NEEDED



In [12]:
# CELL 10: Process all reviews and create tickets
print("="*60)
print("📊 PROCESSING ALL REVIEWS")
print("="*60)

tickets = []
for idx, row in df.iterrows():
    emotion, conf, _ = predict_emotion(row['Review_text'])
    
    if emotion == 'anger/sadness':
        tickets.append({
            'ticket_id': f"TKT-{idx:04d}",
            'review': str(row['Review_text'])[:100],
            'category': row['Category'],
            'rating': row['Rating'],
            'emotion': emotion,
            'confidence': f"{conf:.2%}",
            'priority': 'HIGH'
        })
    elif emotion == 'neutral':
        tickets.append({
            'ticket_id': f"TKT-{idx:04d}",
            'review': str(row['Review_text'])[:100],
            'category': row['Category'],
            'rating': row['Rating'],
            'emotion': emotion,
            'confidence': f"{conf:.2%}",
            'priority': 'MEDIUM'
        })

print(f"\n✅ Total reviews processed: {len(df)}")
print(f"🎫 Tickets created: {len(tickets)}")
print(f"📈 Ticket rate: {len(tickets)/len(df)*100:.1f}%")

📊 PROCESSING ALL REVIEWS

✅ Total reviews processed: 60857
🎫 Tickets created: 8789
📈 Ticket rate: 14.4%


In [13]:
# CELL 11: Save tickets
if len(tickets) > 0:
    tickets_df = pd.DataFrame(tickets)
    tickets_df.to_csv('trained_ai_tickets.csv', index=False)
    print(f"✅ Saved {len(tickets)} tickets to 'trained_ai_tickets.csv'")
    print("\n📋 First 5 tickets:")
    print(tickets_df.head())
else:
    print("No tickets created")

✅ Saved 8789 tickets to 'trained_ai_tickets.csv'

📋 First 5 tickets:
  ticket_id                                             review category  \
0  TKT-0008  The company should give more Bettany backup an...   mobile   
1  TKT-0023  Product not working. Carrier not getting detec...  smartTv   
2  TKT-0031  Worst product!!!! Just becz of rain i got wate...  smartTv   
3  TKT-0033  Phone is ok... But there are lot of problems s...   mobile   
4  TKT-0040  Got pirated copy as there was no hologram and ...    books   

   rating        emotion confidence priority  
0       2  anger/sadness     43.93%     HIGH  
1       1  anger/sadness     68.81%     HIGH  
2       1  anger/sadness     96.97%     HIGH  
3       3  anger/sadness     72.31%     HIGH  
4       1  anger/sadness     81.85%     HIGH  


In [ ]:
# CELL 12: Interactive tester
print("="*60)
print("🎯 INTERACTIVE AI TESTER")
print("="*60)
print("Type 'quit' to exit\n")

while True:
    review = input("📝 Enter customer review: ")
    if review.lower() == 'quit':
        print("\n👋 Goodbye!")
        break
    if review.strip():
        emotion, conf, action = predict_emotion(review)
        print(f"\n🤖 AI Prediction:")
        print(f"   Emotion: {emotion} ({conf:.2%})")
        print(f"   {action}\n")

🎯 INTERACTIVE AI TESTER
Type 'quit' to exit

